# ANRF AISEHack Phase 2 — PM2.5 Pollution Forecasting
## Architecture: Adaptive Fourier Neural Operator (AFNO)

**Setup:** Add the competition dataset as input. The dataset path will be:
`/kaggle/input/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/`

The notebook will:
1. Compute normalization statistics from training data
2. Train an AFNO model with episode-aware loss
3. Run OOM-safe batched inference
4. Save `preds.npy` to `/kaggle/working/`

In [1]:
import os
import gc
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


## Configuration
**Update `DATA_ROOT` below to match your Kaggle dataset path if needed.**

In [2]:
class Config:
    # ── Paths ──────────────────────────────────────────────────────────────
    # This is the standard Kaggle path for this competition dataset.
    # If your dataset slug is different, update DATA_ROOT accordingly.
    DATA_ROOT   = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2'
    WORKING_DIR = '/kaggle/working'

    RAW_DIR  = os.path.join(DATA_ROOT, 'raw')
    TEST_DIR = os.path.join(DATA_ROOT, 'test_in')

    MONTHS = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']

    # ── Features ───────────────────────────────────────────────────────────
    # 13 features selected: core met drivers + key emission tracers
    FEATURES = [
        'cpm25',       # PM2.5 concentration (target / context)
        'q2',          # specific humidity
        't2',          # 2m temperature
        'u10',         # zonal wind
        'v10',         # meridional wind
        'pblh',        # planetary boundary layer height
        'psfc',        # surface pressure
        'swdown',      # shortwave radiation (diurnal cycle proxy)
        'rain',        # precipitation (wet deposition)
        'PM25',        # PM2.5 emission flux
        'NOx',         # NOx emission flux
        'SO2',         # SO2 emission flux
        'NH3',         # NH3 emission flux
    ]
    NUM_FEATURES = len(FEATURES)

    # Emission features are near-zero (~1e-7): use max-scaling
    MAX_SCALE_FEATURES = {'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio'}
    # Heavy-tailed sparse features: log1p + global z-score
    LOG_TRANSFORM_FEATURES = {'rain'}

    # ── Temporal ───────────────────────────────────────────────────────────
    INPUT_HOURS  = 10
    OUTPUT_HOURS = 16
    WINDOW_SIZE  = INPUT_HOURS + OUTPUT_HOURS   # 26

    # ── Spatial ────────────────────────────────────────────────────────────
    H = 140
    W = 124

    # ── Training ───────────────────────────────────────────────────────────
    STRIDE     = 2       # sliding window stride
    BATCH_SIZE = 4
    VAL_SPLIT  = 0.1
    EPOCHS     = 50
    LR         = 3e-4
    WEIGHT_DECAY = 1e-5
    WARMUP_EPOCHS = 3
    PATIENCE   = 7

    # ── AFNO Architecture ──────────────────────────────────────────────────
    EMBED_DIM  = 256
    DEPTH      = 8
    NUM_HEADS  = 4
    MLP_RATIO  = 4.0
    DROP_RATE  = 0.1
    PATCH_SIZE = (4, 4)  # 140/4=35, 124/4=31 → 1085 tokens

    # ── Episode detection ──────────────────────────────────────────────────
    EPISODE_STD_THRESHOLD = 3.0
    EPISODE_MIN_CPM       = 1.0

    # ── Loss weights (mirror competition metric weights) ───────────────────
    W_GLOBAL_SMAPE  = 0.33
    W_EPISODE_CORR  = 0.34
    W_EPISODE_SMAPE = 0.33

    # ── Inference ──────────────────────────────────────────────────────────
    INFERENCE_BATCH_SIZE = 8
    NUM_TEST_SAMPLES     = 218

    SEED = 42


def seed_everything(seed=Config.SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Validate paths
assert os.path.exists(Config.RAW_DIR),  f'Raw dir not found: {Config.RAW_DIR}'
assert os.path.exists(Config.TEST_DIR), f'Test dir not found: {Config.TEST_DIR}'
print('Data paths validated.')

Device: cuda
Data paths validated.


## Step 1 — Normalization Statistics

In [3]:
class NormStats:
    """Per-feature normalization statistics (gridwise or global)."""

    def __init__(self):
        self.mean = {}
        self.std  = {}

    def compute_from_raw(self, raw_dir, months, features):
        print('Computing normalization statistics...')
        for feat in features:

            if feat in Config.MAX_SCALE_FEATURES:
                # Emission features: max-scaling (mean=0, std=global_max)
                global_max = 0.0
                for month in months:
                    d = np.load(os.path.join(raw_dir, month, f'{feat}.npy'), mmap_mode='r')
                    global_max = max(global_max, float(np.max(np.abs(d))))
                self.mean[feat] = np.zeros((Config.H, Config.W), dtype=np.float32)
                self.std[feat]  = np.full((Config.H, Config.W), max(global_max, 1e-15), dtype=np.float32)
                print(f'  {feat} [max-scale]: global_max={global_max:.2e}')

            elif feat in Config.LOG_TRANSFORM_FEATURES:
                # Rain: log1p then global z-score (avoids zero-std at dry grid points)
                s, sq, n = 0.0, 0.0, 0
                for month in months:
                    d = np.load(os.path.join(raw_dir, month, f'{feat}.npy'), mmap_mode='r')
                    x = np.log1p(d.astype(np.float64))
                    s  += x.sum();  sq += (x**2).sum();  n += x.size
                gm  = s / n
                gstd = max(np.sqrt(sq / n - gm**2), 1e-6)
                self.mean[feat] = np.full((Config.H, Config.W), gm,   dtype=np.float32)
                self.std[feat]  = np.full((Config.H, Config.W), gstd, dtype=np.float32)
                print(f'  {feat} [log1p+global_zscore]: mean={gm:.4f}, std={gstd:.4f}')

            else:
                # Standard gridwise z-score via Welford's online algorithm
                count    = 0
                mean_acc = np.zeros((Config.H, Config.W), dtype=np.float64)
                m2_acc   = np.zeros((Config.H, Config.W), dtype=np.float64)
                for month in months:
                    d = np.load(os.path.join(raw_dir, month, f'{feat}.npy'), mmap_mode='r')
                    for t in range(d.shape[0]):
                        x = d[t].astype(np.float64)
                        count  += 1
                        delta   = x - mean_acc
                        mean_acc += delta / count
                        m2_acc  += delta * (x - mean_acc)
                self.mean[feat] = mean_acc.astype(np.float32)
                self.std[feat]  = np.maximum(np.sqrt(m2_acc / max(count-1, 1)).astype(np.float32), 1e-6)
                print(f'  {feat}: mean=[{self.mean[feat].min():.2f},{self.mean[feat].max():.2f}], '
                      f'std=[{self.std[feat].min():.4f},{self.std[feat].max():.2f}]')

    def normalize(self, data, feat_name):
        """Normalize array (..., H, W) using stored stats."""
        if feat_name in Config.LOG_TRANSFORM_FEATURES:
            data = np.log1p(data)
        return (data - self.mean[feat_name]) / self.std[feat_name]

    def denormalize_cpm25(self, data_tensor, device):
        """Denormalize a cpm25 tensor back to µg/m³."""
        m = torch.from_numpy(self.mean['cpm25']).to(device)
        s = torch.from_numpy(self.std['cpm25']).to(device)
        return data_tensor * s + m

    def save(self, path):
        np.savez(path,
                 features=list(self.mean.keys()),
                 **{f'mean_{k}': v for k, v in self.mean.items()},
                 **{f'std_{k}':  v for k, v in self.std.items()})

    def load(self, path):
        d = np.load(path, allow_pickle=True)
        for feat in list(d['features']):
            self.mean[feat] = d[f'mean_{feat}']
            self.std[feat]  = d[f'std_{feat}']


stats_path = os.path.join(Config.WORKING_DIR, 'norm_stats.npz')
norm_stats = NormStats()

if os.path.exists(stats_path):
    print('Loading cached stats...')
    norm_stats.load(stats_path)
else:
    norm_stats.compute_from_raw(Config.RAW_DIR, Config.MONTHS, Config.FEATURES)
    norm_stats.save(stats_path)

gc.collect()
print('Done.')

Computing normalization statistics...
  cpm25: mean=[2.51,274.26], std=[1.4931,247.59]
  q2: mean=[0.00,0.02], std=[0.0008,0.01]
  t2: mean=[255.68,303.37], std=[0.4390,13.23]
  u10: mean=[-2.61,7.09], std=[0.6547,6.79]
  v10: mean=[-3.51,7.61], std=[0.4179,6.19]
  pblh: mean=[262.77,1514.86], std=[86.4051,1172.87]
  psfc: mean=[49109.23,101124.63], std=[98.4053,846.72]
  swdown: mean=[129.73,289.67], std=[214.6863,376.31]
  rain [log1p+global_zscore]: mean=0.0442, std=0.2089
  PM25 [max-scale]: global_max=1.43e-07
  NOx [max-scale]: global_max=7.98e-08
  SO2 [max-scale]: global_max=4.33e-08
  NH3 [max-scale]: global_max=2.09e-08
Done.


## Step 2 — Dataset & DataLoader

In [4]:
class PM25TrainDataset(Dataset):
    """
    Sliding-window dataset over raw monthly data.
    Uses mmap to avoid loading all months into RAM.
    """

    def __init__(self, raw_dir, months, features, norm_stats, stride=Config.STRIDE):
        self.features = features
        self.norm_stats = norm_stats
        self.window = Config.WINDOW_SIZE
        self.samples = []  # list of (month_idx, start_hour)
        self.data_paths = {}  # (month_idx, feat) -> path

        for m_idx, month in enumerate(months):
            T = np.load(
                os.path.join(raw_dir, month, 'cpm25.npy'),
                mmap_mode='r'
            ).shape[0]

            for i in range(0, T - self.window + 1, stride):
                self.samples.append((m_idx, i))

            for feat in features:
                self.data_paths[(m_idx, feat)] = os.path.join(
                    raw_dir, month, f'{feat}.npy'
                )

        print(f'TrainDataset: {len(self.samples)} samples, stride={stride}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        m_idx, start = self.samples[idx]

        # Input: all features for INPUT_HOURS window
        inp_list = []
        for feat in self.features:
            raw = np.load(self.data_paths[(m_idx, feat)], mmap_mode='r')
            chunk = raw[start:start + Config.INPUT_HOURS].astype(np.float32)
            inp_list.append(self.norm_stats.normalize(chunk, feat))

        inputs = np.stack(inp_list, axis=0)  # (F, 10, H, W)

        # Target: cpm25 for OUTPUT_HOURS
        cpm_raw = np.load(self.data_paths[(m_idx, 'cpm25')], mmap_mode='r')

        ts = start + Config.INPUT_HOURS
        target_raw = cpm_raw[
            ts:ts + Config.OUTPUT_HOURS
        ].astype(np.float32)  # (16, H, W)

        target_norm = self.norm_stats.normalize(
            target_raw.copy(), 'cpm25'
        )

        context_raw = cpm_raw[
            start:start + Config.INPUT_HOURS
        ].astype(np.float32)  # (10, H, W)

        return (
            torch.from_numpy(inputs),
            torch.from_numpy(target_norm),
            torch.from_numpy(target_raw),
            torch.from_numpy(context_raw)
        )


class PM25TestDataset(Dataset):
    """
    Memory-mapped test dataset — no data loaded into RAM until accessed.
    """

    def __init__(self, test_dir, features, norm_stats):
        self.features = features
        self.norm_stats = norm_stats
        self.data = {}

        for feat in features:
            self.data[feat] = np.load(
                os.path.join(test_dir, f'{feat}.npy'),
                mmap_mode='c'
            )

        self.n = self.data[features[0]].shape[0]
        print(f'TestDataset: {self.n} samples (mmap)')

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        inp_list = []
        for feat in self.features:
            chunk = self.data[feat][idx].astype(np.float32)  # (10, H, W)
            inp_list.append(self.norm_stats.normalize(chunk, feat))

        return torch.from_numpy(np.stack(inp_list, axis=0))  # (F, 10, H, W)


# Build train / val split
full_ds = PM25TrainDataset(
    Config.RAW_DIR,
    Config.MONTHS,
    Config.FEATURES,
    norm_stats
)

n_val = int(len(full_ds) * Config.VAL_SPLIT)
NUM_WORKERS = 2 if os.path.exists("/kaggle") else 0

n_train = len(full_ds) - n_val

train_ds, val_ds = torch.utils.data.random_split(
    full_ds,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(Config.SEED)
)

print(f'Train: {n_train}  Val: {n_val}')


train_loader = DataLoader(
    train_ds,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=(NUM_WORKERS > 0)
)

val_loader = DataLoader(
    val_ds,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

TrainDataset: 1416 samples, stride=2
Train: 1275  Val: 141


## Step 3 — AFNO Architecture

In [5]:
class PatchEmbed(nn.Module):
    """Fuse temporal dim into channels, then apply 2-D patch projection."""

    def __init__(self, in_channels, embed_dim, patch_size, input_hours):
        super().__init__()
        self.proj = nn.Conv2d(in_channels * input_hours, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        B, C, T, H, W = x.shape
        x = x.reshape(B, C * T, H, W)
        x = self.proj(x)                         # (B, D, Hp, Wp)
        B, D, Hp, Wp = x.shape
        x = x.flatten(2).transpose(1, 2)         # (B, N, D)
        return self.norm(x), Hp, Wp


class AdaptiveFourierBlock(nn.Module):
    """One AFNO block: 2-D FFT → learnable complex weight mixing → IFFT → MLP."""

    def __init__(self, dim, num_heads=4, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        head_dim = dim // num_heads
        self.num_heads = num_heads
        self.norm1 = nn.LayerNorm(dim)
        self.scale = nn.Parameter(torch.ones(1, 1, dim) * 0.02)
        self.bias  = nn.Parameter(torch.zeros(1, 1, dim))
        self.freq_w_re = nn.Parameter(torch.randn(num_heads, head_dim, head_dim) * 0.02)
        self.freq_w_im = nn.Parameter(torch.randn(num_heads, head_dim, head_dim) * 0.02)
        self.sparsity  = 0.01
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, dim), nn.Dropout(drop)
        )

    def forward(self, x, Hp, Wp):
        B, N, D = x.shape
        res = x
        x   = self.norm1(x).reshape(B, Hp, Wp, D)

        xf  = torch.fft.rfft2(x, dim=(1, 2), norm='ortho')   # (B,Hp,Wp/2+1,D)
        bx, hx, wx, dx = xf.shape
        hd = dx // self.num_heads
        xf = xf.reshape(bx, hx, wx, self.num_heads, hd)

        # Learnable complex weight: W @ x_freq
        o_re = (torch.einsum('bhwnd,nde->bhwne', xf.real, self.freq_w_re)
              - torch.einsum('bhwnd,nde->bhwne', xf.imag, self.freq_w_im))
        o_im = (torch.einsum('bhwnd,nde->bhwne', xf.real, self.freq_w_im)
              + torch.einsum('bhwnd,nde->bhwne', xf.imag, self.freq_w_re))

        xf = torch.complex(
            F.softshrink(o_re, self.sparsity),
            F.softshrink(o_im, self.sparsity)
        ).reshape(bx, hx, wx, dx)
        with torch.cuda.amp.autocast(enabled=False):
            x = torch.fft.irfft2(xf.float(), s=(Hp, Wp), dim=(1, 2), norm='ortho')  # (B,Hp,Wp,D)
        x = x.reshape(B, N, D) * self.scale + self.bias + res
        return x + self.mlp(self.norm2(x))


class TemporalProjectionHead(nn.Module):
    """Project token embeddings → (B, T_out, H, W) via linear + unpatchify."""

    def __init__(self, embed_dim, output_hours, patch_size, img_h, img_w):
        super().__init__()
        self.out_t  = output_hours
        self.ph, self.pw = patch_size
        self.Hp = img_h // patch_size[0]
        self.Wp = img_w // patch_size[1]
        out_dim = output_hours * patch_size[0] * patch_size[1]
        self.proj = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim), nn.GELU(),
            nn.Linear(embed_dim, out_dim)
        )

    def forward(self, x):
        B = x.shape[0]
        x = self.proj(x)                                          # (B, N, T_out*Ph*Pw)
        x = x.reshape(B, self.Hp, self.Wp, self.out_t, self.ph, self.pw)
        x = x.permute(0, 3, 1, 4, 2, 5).contiguous()            # (B, T, Hp, Ph, Wp, Pw)
        return x.reshape(B, self.out_t,
                         self.Hp * self.ph, self.Wp * self.pw)   # (B, T, H, W)


class AFNO_PM25(nn.Module):
    """
    Adaptive Fourier Neural Operator for PM2.5 spatiotemporal forecasting.
    Input:  (B, F, T_in,  H, W)  — F features, 10 hours of context
    Output: (B, T_out, H, W)     — 16 hours of cpm25 (normalised)
    """

    def __init__(self, cfg=Config):
        super().__init__()
        self.patch_embed = PatchEmbed(cfg.NUM_FEATURES, cfg.EMBED_DIM,
                                       cfg.PATCH_SIZE, cfg.INPUT_HOURS)
        Hp = cfg.H // cfg.PATCH_SIZE[0]    # 35
        Wp = cfg.W // cfg.PATCH_SIZE[1]    # 31
        self.pos_embed = nn.Parameter(torch.zeros(1, Hp * Wp, cfg.EMBED_DIM))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.blocks = nn.ModuleList([
            AdaptiveFourierBlock(cfg.EMBED_DIM, cfg.NUM_HEADS,
                                  cfg.MLP_RATIO, cfg.DROP_RATE)
            for _ in range(cfg.DEPTH)
        ])
        self.head = TemporalProjectionHead(cfg.EMBED_DIM, cfg.OUTPUT_HOURS,
                                            cfg.PATCH_SIZE, cfg.H, cfg.W)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x, Hp, Wp = self.patch_embed(x)   # (B, N, D)
        x = x + self.pos_embed
        for blk in self.blocks:
            x = blk(x, Hp, Wp)
        return self.head(x)               # (B, T_out, H, W)


model   = AFNO_PM25(Config).to(DEVICE)
n_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_param:,} ({n_param/1e6:.1f}M)')

# Sanity-check shapes
with torch.no_grad():
    dummy = torch.randn(1, Config.NUM_FEATURES, Config.INPUT_HOURS, Config.H, Config.W).to(DEVICE)
    out   = model(dummy)
    assert out.shape == (1, Config.OUTPUT_HOURS, Config.H, Config.W)
    print(f'Forward pass OK: {dummy.shape} → {out.shape}')
del dummy, out
torch.cuda.empty_cache()

Parameters: 5,422,080 (5.4M)
Forward pass OK: torch.Size([1, 13, 10, 140, 124]) → torch.Size([1, 16, 140, 124])


/tmp/ipykernel_24/635602522.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/tmp/ipykernel_24/635602522.py:60: UserWarning: Casting complex values to real discards the imaginary part (Triggered internally at /pytorch/aten/src/ATen/native/Copy.cpp:308.)
  x = torch.fft.irfft2(xf.float(), s=(Hp, Wp), dim=(1, 2), norm='ortho')  # (B,Hp,Wp,D)


## Step 4 — Episode-Aware Loss Function

In [6]:
class EpisodeAwareLoss(nn.Module):
    """
    Composite loss that directly optimises the three competition metrics:
      1. Global SMAPE          (all grid points, all timesteps)
      2. Episode SMAPE         (episodic grid points only)
      3. Episode Pearson Corr  (episodic grid points — rewarded, not penalised)
    Plus an MSE term in normalised space for stable gradient flow early in training.

    Episode mask: grid points where the STL-residual proxy (temporal z-score
    relative to context window) exceeds 3σ AND raw cpm25 > 1 µg/m³.
    """

    def __init__(self, w_global=Config.W_GLOBAL_SMAPE,
                 w_ep_corr=Config.W_EPISODE_CORR,
                 w_ep_smape=Config.W_EPISODE_SMAPE, eps=1e-8):
        super().__init__()
        self.w_global   = w_global
        self.w_ep_corr  = w_ep_corr
        self.w_ep_smape = w_ep_smape
        self.eps        = eps

    @staticmethod
    def smape(pred, tgt, eps=1e-8):
        return torch.abs(pred - tgt) / (0.5 * (torch.abs(pred) + torch.abs(tgt)) + eps)

    def episode_mask(self, tgt_raw, ctx_raw):
        """Binary mask (B, T_out, H, W): 1 where an episode occurs."""
        base_mean = ctx_raw.mean(dim=1, keepdim=True)          # (B,1,H,W)
        base_std  = ctx_raw.std(dim=1, keepdim=True) + self.eps
        z = (tgt_raw - base_mean) / base_std
        return ((z > Config.EPISODE_STD_THRESHOLD) &
                (tgt_raw > Config.EPISODE_MIN_CPM)).float()

    def pearson_on_episodes(self, pred_raw, tgt_raw, mask):
        """Mean Pearson correlation across timesteps on episodic points."""
        B, T, H, W = pred_raw.shape
        corr_sum, valid = 0.0, 0
        for t in range(T):
            m = mask[:, t].reshape(-1)         # (B*H*W,)
            if m.sum() < 2: continue
            idx = m > 0.5
            p = pred_raw[:, t].reshape(-1)[idx]
            y = tgt_raw[:,  t].reshape(-1)[idx]
            pc = p - p.mean();  yc = y - y.mean()
            cov  = (pc * yc).sum()
            denom = (pc.pow(2).sum().sqrt() * yc.pow(2).sum().sqrt() + self.eps)
            corr_sum += cov / denom
            valid += 1
        if valid == 0:
            return torch.tensor(0.0, device=pred_raw.device)
        return corr_sum / valid

    def forward(self, pred_norm, tgt_norm, tgt_raw, ctx_raw, norm_stats):
        dev = pred_norm.device
        m_t = torch.from_numpy(norm_stats.mean['cpm25']).to(dev)
        s_t = torch.from_numpy(norm_stats.std['cpm25']).to(dev)
        pred_raw = F.relu(pred_norm * s_t + m_t)   # denorm + clamp ≥ 0

        # Global SMAPE
        l_global = self.smape(pred_raw, tgt_raw).mean()

        # Episode mask
        mask  = self.episode_mask(tgt_raw, ctx_raw)
        n_ep  = mask.sum()

        if n_ep > 0:
            ep_smape_vals = self.smape(pred_raw, tgt_raw) * mask
            l_ep_smape   = ep_smape_vals.sum() / (n_ep + self.eps)
            ep_corr      = self.pearson_on_episodes(pred_raw, tgt_raw, mask)
            l_ep_corr    = 1.0 - ep_corr      # minimise → maximise correlation
        else:
            l_ep_smape = torch.tensor(0.0, device=dev)
            l_ep_corr  = torch.tensor(0.0, device=dev)

        mse = F.mse_loss(pred_norm, tgt_norm)   # normalised-space regulariser

        total = (self.w_global   * l_global
               + self.w_ep_smape * l_ep_smape
               + self.w_ep_corr  * l_ep_corr
               + 0.5             * mse)

        metrics = dict(
            global_smape  = l_global.item(),
            episode_smape = l_ep_smape.item() if isinstance(l_ep_smape, torch.Tensor) else 0.0,
            episode_corr  = (1 - l_ep_corr.item()) if isinstance(l_ep_corr, torch.Tensor) else 0.0,
            mse           = mse.item(),
            n_episodes    = n_ep.item(),
        )
        return total, metrics


criterion = EpisodeAwareLoss()
print('Loss function ready.')

Loss function ready.


## Step 5 — Training Loop with AMP

In [7]:
def cosine_schedule_with_warmup(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs  * steps_per_epoch
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, norm_stats, device, ep):
    model.train()
    tot, n = 0.0, 0
    acc = dict(global_smape=0, episode_smape=0, episode_corr=0, mse=0)
    for i, (inp, tgt_n, tgt_r, ctx_r) in enumerate(loader):
        inp   = inp.to(device, non_blocking=True)
        tgt_n = tgt_n.to(device, non_blocking=True)
        tgt_r = tgt_r.to(device, non_blocking=True)
        ctx_r = ctx_r.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(dtype=torch.float16):
            pred = model(inp)
            loss, metrics = criterion(pred.float(), tgt_n, tgt_r, ctx_r, norm_stats)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        tot += loss.item(); n += 1
        for k in acc: acc[k] += metrics.get(k, 0)
        if i % 100 == 0:
            print(f'  [{ep}][{i}/{len(loader)}] loss={loss.item():.4f} '
                  f'smape={metrics["global_smape"]:.4f} ep_corr={metrics["episode_corr"]:.4f}')
    for k in acc: acc[k] /= max(n, 1)
    return tot / max(n, 1), acc


@torch.no_grad()
def validate(model, loader, criterion, norm_stats, device):
    model.eval()
    tot, n = 0.0, 0
    acc = dict(global_smape=0, episode_smape=0, episode_corr=0, mse=0)
    for inp, tgt_n, tgt_r, ctx_r in loader:
        inp   = inp.to(device, non_blocking=True)
        tgt_n = tgt_n.to(device, non_blocking=True)
        tgt_r = tgt_r.to(device, non_blocking=True)
        ctx_r = ctx_r.to(device, non_blocking=True)
        with autocast(dtype=torch.float16):
            pred = model(inp)
            loss, metrics = criterion(pred.float(), tgt_n, tgt_r, ctx_r, norm_stats)
        tot += loss.item(); n += 1
        for k in acc: acc[k] += metrics.get(k, 0)
    for k in acc: acc[k] /= max(n, 1)
    return tot / max(n, 1), acc


optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR,
                               weight_decay=Config.WEIGHT_DECAY)
scheduler = cosine_schedule_with_warmup(
    optimizer, Config.WARMUP_EPOCHS, Config.EPOCHS, len(train_loader)
)
scaler    = GradScaler()

best_val_loss  = float('inf')
patience_cnt   = 0
ckpt_path      = os.path.join(Config.WORKING_DIR, 'best_model.pt')

print(f'Starting training for up to {Config.EPOCHS} epochs...')
print(f'Steps per epoch: {len(train_loader)}')
print('-' * 60)

for epoch in range(1, Config.EPOCHS + 1):
    t0 = time.time()
    train_loss, train_m = train_epoch(model, train_loader, criterion, optimizer,
                                       scheduler, scaler, norm_stats, DEVICE, epoch)
    val_loss,   val_m   = validate(model, val_loader, criterion, norm_stats, DEVICE)
    elapsed = time.time() - t0

    print(f'\nEpoch {epoch}/{Config.EPOCHS}  ({elapsed:.0f}s)')
    print(f'  Train: loss={train_loss:.4f}  smape={train_m["global_smape"]:.4f}  '
          f'ep_smape={train_m["episode_smape"]:.4f}  ep_corr={train_m["episode_corr"]:.4f}')
    print(f'  Val:   loss={val_loss:.4f}    smape={val_m["global_smape"]:.4f}  '
          f'ep_smape={val_m["episode_smape"]:.4f}  ep_corr={val_m["episode_corr"]:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_cnt  = 0
        torch.save(dict(epoch=epoch,
                        model_state_dict=model.state_dict(),
                        val_loss=val_loss,
                        val_metrics=val_m), ckpt_path)
        print(f'  ✓ Best model saved  (val_loss={val_loss:.4f})')
    else:
        patience_cnt += 1
        print(f'  No improvement ({patience_cnt}/{Config.PATIENCE})')
        if patience_cnt >= Config.PATIENCE:
            print('Early stopping.')
            break

print('Training complete.')

Starting training for up to 50 epochs...
Steps per epoch: 318
------------------------------------------------------------


/tmp/ipykernel_24/2695285040.py:64: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()
/tmp/ipykernel_24/2695285040.py:22: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):
/tmp/ipykernel_24/635602522.py:55: UserWarning: ComplexHalf support is experimental and many operators don't support it yet. (Triggered internally at /pytorch/aten/src/ATen/EmptyTensor.cpp:54.)
  xf = torch.complex(
/tmp/ipykernel_24/635602522.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


  [1][0/318] loss=1.0784 smape=0.8388 ep_corr=0.3672
  [1][100/318] loss=0.8776 smape=0.5905 ep_corr=0.7619
  [1][200/318] loss=0.4550 smape=0.4150 ep_corr=0.9181
  [1][300/318] loss=0.4210 smape=0.3582 ep_corr=0.9124


/tmp/ipykernel_24/2695285040.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):



Epoch 1/50  (129s)
  Train: loss=0.6939  smape=0.5045  ep_smape=0.4687  ep_corr=0.7835
  Val:   loss=0.4632    smape=0.3756  ep_smape=0.3558  ep_corr=0.8801
  ✓ Best model saved  (val_loss=0.4632)
  [2][0/318] loss=0.3595 smape=0.3343 ep_corr=0.9338
  [2][100/318] loss=0.3494 smape=0.3290 ep_corr=0.8972
  [2][200/318] loss=0.5208 smape=0.4268 ep_corr=0.8060
  [2][300/318] loss=0.4017 smape=0.3689 ep_corr=0.9277

Epoch 2/50  (113s)
  Train: loss=0.4286  smape=0.3715  ep_smape=0.3445  ep_corr=0.9037
  Val:   loss=0.4396    smape=0.3540  ep_smape=0.3612  ep_corr=0.8826
  ✓ Best model saved  (val_loss=0.4396)
  [3][0/318] loss=0.4351 smape=0.4465 ep_corr=0.9562
  [3][100/318] loss=0.3178 smape=0.3263 ep_corr=0.9585
  [3][200/318] loss=0.4343 smape=0.3942 ep_corr=0.9230
  [3][300/318] loss=0.3927 smape=0.3648 ep_corr=0.8800

Epoch 3/50  (99s)
  Train: loss=0.4111  smape=0.3619  ep_smape=0.3350  ep_corr=0.9103
  Val:   loss=0.4176    smape=0.3540  ep_smape=0.3151  ep_corr=0.8935
  ✓ Best mo

## Step 6 — OOM-Safe Inference

In [8]:
# ── Load best checkpoint ──────────────────────────────────────────────────
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best model: epoch={ckpt["epoch"]}  val_loss={ckpt["val_loss"]:.4f}')
print(f'Val metrics: {ckpt["val_metrics"]}')

# ── Free training RAM ────────────────────────────────────────────────────

# ── Inference ────────────────────────────────────────────────────────────
model.eval()

test_ds     = PM25TestDataset(Config.TEST_DIR, Config.FEATURES, norm_stats)
test_loader = DataLoader(test_ds,
                         batch_size=Config.INFERENCE_BATCH_SIZE,
                         shuffle=False, num_workers=0,
                         pin_memory=torch.cuda.is_available())

m_t = torch.from_numpy(norm_stats.mean['cpm25']).to(DEVICE)   # (H, W)
s_t = torch.from_numpy(norm_stats.std['cpm25']).to(DEVICE)    # (H, W)

# Pre-allocate output — final shape (218, 140, 124, 16)
preds = np.zeros((Config.NUM_TEST_SAMPLES, Config.H, Config.W, Config.OUTPUT_HOURS),
                  dtype=np.float32)

idx = 0
with torch.no_grad():
    for batch_inp in test_loader:
        batch_inp = batch_inp.to(DEVICE, non_blocking=True)
        bs = batch_inp.shape[0]

        with autocast(dtype=torch.float16):
            pred_norm = model(batch_inp).float()

        pred_raw = F.relu(pred_norm * s_t + m_t)          # denorm, clamp ≥ 0
        pred_np  = pred_raw.cpu().numpy()                   # (B, T_out, H, W)
        pred_np  = np.transpose(pred_np, (0, 2, 3, 1))     # (B, H, W, T_out)
        preds[idx : idx + bs] = pred_np
        idx += bs

        if idx % 50 == 0 or idx == Config.NUM_TEST_SAMPLES:
            print(f'  {idx}/{Config.NUM_TEST_SAMPLES} samples done')

        del batch_inp, pred_norm, pred_raw
        torch.cuda.empty_cache()

# ── Save submission ──────────────────────────────────────────────────────
out_path = os.path.join(Config.WORKING_DIR, 'preds.npy')
np.save(out_path, preds)

print(f'\nSaved: {out_path}')
print(f'Shape: {preds.shape}  (expected: (218, 140, 124, 16))')
assert preds.shape == (218, 140, 124, 16), f'Shape mismatch! {preds.shape}'
print(f'Value range: [{preds.min():.2f}, {preds.max():.2f}]  Mean: {preds.mean():.2f}')
print('\n✓ Submission ready: /kaggle/working/preds.npy')

Loaded best model: epoch=50  val_loss=0.1860
Val metrics: {'global_smape': 0.20363914966583252, 'episode_smape': 0.17074158994687927, 'episode_corr': 0.9721211344003677, 'mse': 0.10601720545026991}
TestDataset: 218 samples (mmap)


/tmp/ipykernel_24/3403912779.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):
/tmp/ipykernel_24/635602522.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


  200/218 samples done
  218/218 samples done

Saved: /kaggle/working/preds.npy
Shape: (218, 140, 124, 16)  (expected: (218, 140, 124, 16))
Value range: [0.00, 1215.89]  Mean: 36.24

✓ Submission ready: /kaggle/working/preds.npy
